<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# Python & AI in Asset Management
## Chapter 4 · Mean–Variance Portfolio Theory

&copy; Dr. Yves J. Hilpisch<br>
AI-Powered by GPT 5.1<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh


## Notebook Goals

This notebook expands Chapter 4 with executable workflows:

- compute mean/variance inputs directly from <code>pyaiam_eod.csv</code>,
- simulate thousands of random portfolios to visualize the efficient frontier,
- identify the global minimum-variance and maximum Sharpe portfolios, and
- visualize portfolio compositions and frontier diagnostics.


### Getting Help While Optimizing
- **Chapter 2** for covariance/statistics refreshers.
- **Appendix B** for NumPy/pandas math patterns.
- **Appendix A** for the derivations behind mean–variance formulas.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

plt.style.use("seaborn-v0_8")
plt.rcParams.update({"font.family": "serif", "figure.dpi": 300})

DATA_PATH = Path("../data/pyaiam_eod.csv")
if not DATA_PATH.exists():
    DATA_PATH = "https://hilpisch.com/pyaiam_eod.csv"


## 1. Prepare Returns and Moments

We reuse the Chapter 3 data utilities to compute annualized means and covariances for the six-asset universe.

In [ ]:
prices = pd.read_csv(DATA_PATH, parse_dates=["Date"]).set_index("Date").sort_index()
prices = prices.ffill()
log_rets = np.log(prices / prices.shift(1)).dropna()
assets = ["AAPL", "NVDA", "JPM", "SPY", "GLD", "TLT"]
log_rets = log_rets[assets]
exp_returns = log_rets.mean() * 252
cov_matrix = log_rets.cov() * 252
exp_returns

### 1.1 Portfolio Helper Functions

The helper below keeps return/volatility calculations consistent anywhere in the notebook.

In [ ]:
def portfolio_stats(weights: np.ndarray, mean_vec: np.ndarray, cov_mat: np.ndarray) -> tuple[float, float]:
    port_ret = weights @ mean_vec
    port_vol = np.sqrt(weights @ cov_mat @ weights)
    return port_ret, port_vol

risk_free = 0.02

## 2. Monte Carlo Frontier Approximation

Random Dirichlet weights give quick intuition about the feasible return-risk region before formal optimization.

In [ ]:
rng = np.random.default_rng(7)
n_ports = 8000
weights = rng.dirichlet(np.ones(len(assets)), size=n_ports)
port_metrics = np.array([portfolio_stats(w, exp_returns.values, cov_matrix.values)
for w in weights])
port_returns = port_metrics[:, 0]
port_vols = port_metrics[:, 1]
sharpe = (port_returns - risk_free) / port_vols
fig, ax = plt.subplots(figsize=(12, 6))
sc = ax.scatter(port_vols, port_returns, c=sharpe, cmap="viridis", s=6)
ax.set_xlabel("Volatility")
ax.set_ylabel("Expected return")
ax.set_title("Monte Carlo Portfolio Cloud")
fig.colorbar(sc, label="Sharpe ratio")
plt.show()

## 3. Closed-Form GMV and Max-Sharpe Portfolios

Using the textbook matrix formula saves us from iterative solvers for unconstrained GMV and tangency portfolios.

In [ ]:
cov_inv = np.linalg.inv(cov_matrix.values)
ones = np.ones(len(assets))
A = ones @ cov_inv @ ones
B = ones @ cov_inv @ exp_returns.values
gmv_weights = (cov_inv @ ones) / A
gmv_ret, gmv_vol = portfolio_stats(gmv_weights, exp_returns.values,
cov_matrix.values)
ms_weights = (cov_inv @ (exp_returns.values - risk_free * ones))
ms_weights = ms_weights / np.sum(ms_weights)
ms_ret, ms_vol = portfolio_stats(ms_weights, exp_returns.values, cov_matrix.values)
pd.DataFrame(
    {
        "gmv": gmv_weights,
        "max_sharpe": ms_weights,
    },
    index=assets,
)

### 3.1 Plot Frontier and Distinguished Portfolios

Plotting both Monte Carlo samples and distinguished portfolios makes the frontier intuition visual.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
ax.scatter(port_vols, port_returns, c="lightgray", s=6, alpha=0.5)
ax.scatter(gmv_vol, gmv_ret, color="tomato", label="GMV", s=80)
ax.scatter(ms_vol, ms_ret, color="navy", label="Max Sharpe", s=80)
ax.set_xlabel("Volatility")
ax.set_ylabel("Expected return")
ax.set_title("Efficient Frontier Highlights")
ax.legend()
plt.show()

## 4. Portfolio Composition Dashboard

Bar charts let stakeholders compare weight distributions side-by-side.

In [ ]:
weights_df = pd.DataFrame(
    {
        "GMV": gmv_weights,
        "Max Sharpe": ms_weights,
    },
    index=assets,
)
weights_df

In [ ]:
ax = weights_df.plot(kind="bar", figsize=(12, 6))
ax.set_ylabel("Weight")
ax.set_title("GMV vs. Max-Sharpe Weights")
plt.show()

## 5. Sensitivity: Shrinking Expected Returns

Shrinking expected returns mimics Bayesian or robust adjustments without changing covariances.

In [ ]:
shrink_factors = np.linspace(0.2, 1.0, 5)
rows = []
for shrink in shrink_factors:
    shrunk_mean = exp_returns.values * shrink
    w = cov_inv @ (shrunk_mean - risk_free * ones)
    w /= np.sum(w)
    ret, vol = portfolio_stats(w, shrunk_mean, cov_matrix.values)
    rows.append({"shrink": shrink, "ret": ret, "vol": vol})
pd.DataFrame(rows)

## 6. Exercises
### Exercise 1 – Frontier Grid
Solve for efficient portfolios across a grid of target returns using Lagrange multipliers.
<details><summary>Hint</summary>
Use the closed-form coefficients \(A, B, C\) to compute weights for each \(ar r\) target.
</details>

### Exercise 2 – Constrained Optimization
Recompute GMV weights with a long-only constraint (e.g., via <code>scipy.optimize.minimize</code>).
<details><summary>Hint</summary>
Parameterize weights with simplex constraints and pass bounds=(0,1).
</details>

### Exercise 3 – Stress Scenario
Shock <code>exp_returns</code> by subtracting 3 percentage points from equities only and re-plot the frontier.
<details><summary>Hint</summary>
Modify the mean vector selectively using boolean indexing.
</details>


## 7. Takeaways for Chapter 4
- Portfolio statistics map cleanly from theory to NumPy.
- Monte Carlo clouds provide intuition before solving constrained optimizations.
- Sensitivity tests (e.g., shrinking means) reveal how fragile optimized weights can be.


<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">